# AI MCQ Multi-Agent Pipeline — Kaggle Submission Edition

Kiến trúc Đồ thị đa tác tử (Multi-Agent Graph) kết hợp Unsloth 4-bit, Python Sandbox thực thi code và Majority Voting.
Được đóng gói tự động từ cấu trúc modular của dự án.


### 1. Cài đặt Unsloth & Các thư viện bổ trợ


In [ ]:
# Cài đặt Unsloth tương thích tốt nhất với môi trường PyTorch trên Kaggle
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q pandas sympy pydantic pydantic-settings python-dotenv PyYAML tqdm


### 2. Cấu hình mặc định (settings.yaml)


In [ ]:
import os
os.makedirs('config', exist_ok=True)
with open('config/settings.yaml', 'w', encoding='utf-8') as f:
    f.write('''app:
  name: "HackAIthon2026_MultiAgent"
  environment: "production"
  debug: false

llm:
  model_name: "google/gemma-4-E4B-it"
  max_seq_length: 4096
  load_in_4bit: true

agents:
  router:
    temperature: 0.0
    max_new_tokens: 32
    batch_size: 10
  fast_qa:
    temperature: 0.0
    max_new_tokens: 128
    batch_size: 10
  reading:
    temperature: 0.1
    max_new_tokens: 384
    batch_size: 10
  coder:
    temperature: 0.0
    max_new_tokens: 512
    batch_size: 8
  correction:
    temperature: 0.1
    max_new_tokens: 512
    batch_size: 8
  fallback:
    temperature: 0.2
    max_new_tokens: 384
  voting:
    temperature: 0.3
    max_new_tokens: 256
    num_runs: 3
    batch_size: 2

sandbox:
  timeout_sec: 5
  max_retries: 2

paths:
  data_dir: "/data"
  output_dir: "/output"
  checkpoint_file: "pipeline_checkpoint.json"
''')
print('Đã khởi tạo config/settings.yaml thành công.')


### 3. Mã nguồn Hệ thống Multi-Agent


In [ ]:
# ===========================================================================
# MODULE: utils/logger.py
# ===========================================================================
import logging
import logging.config
from pathlib import Path
import yaml

_PROJECT_ROOT = Path(__file__).parent.parent.parent
_LOG_FILE = _PROJECT_ROOT / "config" / "logging.yaml"
_LOGGING_DIR = _PROJECT_ROOT / "output"


def setup_logging(config_path: Path = _LOG_FILE) -> None:
    """Configure logging from a YAML config file."""
    if not config_path.exists():
        raise FileNotFoundError("logging.yaml config file not found")

    with open(config_path, "r", encoding="utf-8") as f:
        logging_config = yaml.safe_load(f)

    # Động hóa đường dẫn file log để luôn trỏ vào thư mục project_root/output tuyệt đối
    if "handlers" in logging_config and "file" in logging_config["handlers"]:
        log_file_path = _LOGGING_DIR / "pipeline.log"
        _LOGGING_DIR.mkdir(parents=True, exist_ok=True)
        logging_config["handlers"]["file"]["filename"] = str(log_file_path)

    logging.config.dictConfig(logging_config)
    logging.info("Read logging.yaml successfully")


# Initialize logging on module import
setup_logging()

logger = logging.getLogger("HackAIthon_Agent")

logging.info("Initialized logger successfully")


# ===========================================================================
# MODULE: core/config.py
# ===========================================================================
from pathlib import Path
from pydantic import BaseModel
from pydantic_settings import BaseSettings, SettingsConfigDict
import yaml
import logging

logger = logging.getLogger("HackAIthon_Agent")

_PROJECT_ROOT = Path(__file__).parent.parent.parent
_SETTING_FILE = _PROJECT_ROOT / "config" / "settings.yaml"


class AppConfig(BaseModel):
    name: str
    environment: str
    debug: bool


class LlmConfig(BaseModel):
    model_name: str
    max_seq_length: int
    load_in_4bit: bool


class AgentParams(BaseModel):
    temperature: float
    max_new_tokens: int = 256
    batch_size: int = 10


class VotingParams(BaseModel):
    temperature: float
    max_new_tokens: int = 256
    num_runs: int = 3
    batch_size: int = 2


class AgentsConfig(BaseModel):
    router: AgentParams
    fast_qa: AgentParams
    reading: AgentParams
    coder: AgentParams
    correction: AgentParams
    fallback: AgentParams
    voting: VotingParams


class SandboxConfig(BaseModel):
    timeout_sec: int
    max_retries: int


class PathsConfig(BaseModel):
    data_dir: str
    output_dir: str
    checkpoint_file: str


class Settings(BaseSettings):
    app: AppConfig
    llm: LlmConfig
    agents: AgentsConfig
    sandbox: SandboxConfig
    paths: PathsConfig

    model_config = SettingsConfigDict(
        env_file=".env", env_file_encoding="utf-8", extra="ignore", case_sensitive=True
    )


def load_settings() -> Settings:
    if not _SETTING_FILE.exists():
        raise FileNotFoundError(f"settings.yaml not found at {_SETTING_FILE}")

    with open(_SETTING_FILE, "r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)

    return Settings(**raw)


try:
    settings = load_settings()
    logger.info("Settings loaded: model=%s, seq_len=%d", settings.llm.model_name, settings.llm.max_seq_length)
except Exception as e:
    logger.error("Failed to load settings: %s", e)
    raise


# ===========================================================================
# MODULE: agents/state.py
# ===========================================================================
"""GraphState — Trạng thái toàn cục luân chuyển qua đồ thị đa tác tử."""

from typing import List, TypedDict


class GraphState(TypedDict):
    questions: List[dict]        # [{qid, question, choices}, ...]
    choice_counts: List[int]     # Số phương án mỗi câu (4, 5, ...)
    routes: List[str]            # FAST_QA, READING, CODEABLE
    execution_codes: List[str]   # Code Python sinh ra (luồng CODEABLE)
    sandbox_results: List[dict]  # {success, stdout, stderr}
    final_answers: List[str]     # Đáp án thô cuối cùng của từng Node


LETTER_MAP = {i: chr(65 + i) for i in range(26)}


def format_choices(choices: list) -> str:
    """Format danh sách lựa chọn thành chuỗi A. xxx\\nB. yyy..."""
    return "\n".join(f"{LETTER_MAP[i]}. {c}" for i, c in enumerate(choices))


def init_state(questions: list) -> GraphState:
    """Khởi tạo GraphState mới từ danh sách câu hỏi."""
    n = len(questions)
    return GraphState(
        questions=questions,
        choice_counts=[len(q["choices"]) for q in questions],
        routes=[""] * n,
        execution_codes=[""] * n,
        sandbox_results=[{}] * n,
        final_answers=[""] * n,
    )


# ===========================================================================
# MODULE: tools/python_sandbox.py
# ===========================================================================
"""Python Sandbox — Chạy code cách ly, trích xuất số, so khớp đáp án."""

import os
import re
import subprocess
import sys
from typing import List, Optional, Tuple



def execute_code(code_string: str, timeout: int = 5) -> dict:
    """Chạy code Python trong subprocess cách ly với timeout."""
    temp_file = "temp_sandbox.py"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(code_string)
    try:
        result = subprocess.run(
            [sys.executable, temp_file],
            capture_output=True, text=True, timeout=timeout,
        )
        return {
            "success": result.returncode == 0,
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
        }
    except subprocess.TimeoutExpired:
        return {
            "success": False,
            "stdout": "",
            "stderr": f"Lỗi: Quá thời gian thực thi (>{timeout} giây).",
        }
    finally:
        if os.path.exists(temp_file):
            try:
                os.remove(temp_file)
            except OSError:
                pass


def extract_numbers_from_text(text: str) -> List[float]:
    """Trích xuất tất cả số (kể cả phân số) từ chuỗi văn bản."""
    text = text.replace(",", "")

    # Bắt phân số: 1/2, -3/4, ...
    fraction_pattern = r'(-?\b\d+)/(\d+\b)'
    fractions = re.findall(fraction_pattern, text)
    nums = []
    for num, denom in fractions:
        try:
            nums.append(float(num) / float(denom))
        except ZeroDivisionError:
            pass

    # Xoá phân số khỏi text trước khi bắt số thập phân
    text_clean = re.sub(fraction_pattern, "", text)

    # Bắt số thập phân: 3.14, -0.5, 100, ...
    decimal_pattern = r'-?\b\d+\.?\d*'
    for n in re.findall(decimal_pattern, text_clean):
        try:
            nums.append(float(n))
        except ValueError:
            pass

    return nums


def is_numeric_choices(choices: list) -> bool:
    """Kiểm tra tất cả các phương án có chứa đúng 1 số không."""
    for choice in choices:
        nums = extract_numbers_from_text(choice)
        if len(nums) != 1:
            return False
    return True


def clean_text_for_match(text: str) -> str:
    """Xoá ký tự đặc biệt để so khớp chuỗi ký tự."""
    text = text.lower().strip()
    return re.sub(r'[\$\{\}\\\^\_\(\)\*\/\+\-\=\s\[\]\,\.]', '', text)


def find_closest_choice_info(
    python_output: str, choices: list
) -> Tuple[Optional[str], float, Optional[float]]:
    """
    So khớp kết quả Python Sandbox với các phương án trắc nghiệm.
    Trả về: (chữ_cái_đáp_án, sai_số_tương_đối, giá_trị_số_thô)
    """
    # Bước 1: Thử so khớp chuỗi ký tự (cho đáp án biểu thức SymPy)
    py_clean = clean_text_for_match(python_output)
    if py_clean and bool(re.search(r'[a-z]', py_clean)):
        for i, choice in enumerate(choices):
            choice_clean = clean_text_for_match(choice)
            if (py_clean == choice_clean
                    or py_clean in choice_clean
                    or choice_clean in py_clean):
                if len(py_clean) >= 2 or py_clean == choice_clean:
                    return LETTER_MAP[i], 0.0, None

    # Bước 2: So khớp số
    if not is_numeric_choices(choices):
        return None, float("inf"), None

    code_nums = extract_numbers_from_text(python_output)
    if not code_nums:
        return None, float("inf"), None

    target_val = code_nums[-1]
    best_letter, min_diff = None, float("inf")

    for i, choice in enumerate(choices):
        for val in extract_numbers_from_text(choice):
            denom = max(abs(val), 1.0)
            diff = abs(target_val - val) / denom
            if diff < min_diff:
                min_diff = diff
                best_letter = LETTER_MAP[i]

    return best_letter, min_diff, target_val


# ===========================================================================
# MODULE: pipeline/dynamic_mapper.py
# ===========================================================================
"""Dynamic Mapper — Trích xuất đáp án từ output LLM thô, chuẩn hoá về A/B/C/D."""

import json
import re
from typing import List


import logging
logger = logging.getLogger("HackAIthon_Agent")


def parse_answer_to_letter(text: str) -> str:
    """
    Trích xuất chữ cái đáp án từ output LLM thô (7 tầng fallback).
    Xử lý được: JSON, SUCCESS_MATCH, markdown bold, tiếng Việt, tiếng Anh, chữ cái đơn.
    """
    if not text:
        return "A"

    # Tầng 0: Trạng thái SUCCESS_MATCH từ Sandbox
    if text.startswith("SUCCESS_MATCH:"):
        parts = text.split(":")
        if len(parts) >= 2:
            letter = parts[1].strip().upper()
            if len(letter) == 1 and letter in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
                return letter

    # Tầng 1: Nếu text là chữ cái đơn lẻ
    text_stripped = text.strip().upper()
    if len(text_stripped) == 1 and text_stripped in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        return text_stripped

    # Tầng 2: Parse JSON (cả trong và ngoài khối ```json```)
    text_clean = text.strip()
    json_match = re.search(r'```json\s*(.*?)\s*```', text_clean, re.DOTALL)
    if json_match:
        text_clean = json_match.group(1).strip()
    else:
        start_idx = text_clean.find("{")
        end_idx = text_clean.rfind("}")
        if start_idx != -1 and end_idx != -1:
            text_clean = text_clean[start_idx: end_idx + 1]

    try:
        data = json.loads(text_clean)
        if isinstance(data, dict):
            for key in ["answer", "choice", "result", "prediction"]:
                ans = data.get(key, "").strip().upper()
                if ans and len(ans) == 1 and ans in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
                    return ans
    except Exception:
        pass

    # Tầng 3: Markdown bold **A**
    match = re.search(r'\*\*([A-Z])\*\*', text)
    if match:
        return match.group(1)

    # Tầng 4: Tiếng Việt — "Đáp án là A" / "đáp án: B"
    match = re.search(r'[đĐ]áp\s*án\s*(?:là|:)?\s*([A-Z])\b', text)
    if match:
        return match.group(1)

    # Tầng 5: Tiếng Anh — "answer is A" / "Answer: B"
    match = re.search(r'[aA]nswer\s*(?:is|:)?\s*([A-Z])\b', text)
    if match:
        return match.group(1)

    # Tầng 6: Chữ cái viết hoa đứng riêng cuối cùng
    matches = re.findall(r'\b([A-Z])\b', text)
    if matches:
        return matches[-1]

    # Tầng 7: Bất kỳ chữ cái viết hoa nào cuối cùng
    matches = re.findall(r'([A-Z])', text)
    if matches:
        return matches[-1]

    return "A"


def map_final_answers(state: dict) -> dict:
    """
    Duyệt qua tất cả final_answers, chuẩn hoá về chữ cái hợp lệ.
    Đảm bảo không vượt quá số phương án thực tế của câu hỏi.
    """
    for i in range(len(state["questions"])):
        raw_answer = state["final_answers"][i]
        num_choices = state["choice_counts"][i]

        # Bước 1: Trích xuất chữ cái
        letter = parse_answer_to_letter(raw_answer)

        # Bước 2: Kiểm tra giới hạn
        idx = ord(letter) - ord("A")
        max_idx = num_choices - 1
        if idx > max_idx:
            letter = "A"  # Fallback an toàn

        state["final_answers"][i] = letter

    return state


# ===========================================================================
# MODULE: pipeline/checkpointing.py
# ===========================================================================
"""Checkpointing — Lưu/nạp trạng thái pipeline chống sập."""

import json
import os


import logging
logger = logging.getLogger("HackAIthon_Agent")


def _get_checkpoint_path() -> str:
    output_dir = settings.paths.output_dir
    os.makedirs(output_dir, exist_ok=True)
    return os.path.join(output_dir, settings.paths.checkpoint_file)


def save_checkpoint(state: GraphState):
    """Ghi đè toàn bộ state hiện tại xuống ổ đĩa."""
    path = _get_checkpoint_path()
    with open(path, "w", encoding="utf-8") as f:
        json.dump({
            "routes": state["routes"],
            "execution_codes": state["execution_codes"],
            "sandbox_results": state["sandbox_results"],
            "final_answers": state["final_answers"],
        }, f, ensure_ascii=False, indent=2)


def load_checkpoint(state: GraphState) -> GraphState:
    """
    Nạp checkpoint nếu có. Ghi đè lên state hiện tại
    (giữ nguyên questions và choice_counts từ dữ liệu mới).
    """
    path = _get_checkpoint_path()
    if not os.path.exists(path):
        logger.info("[Checkpoint] Không có checkpoint cũ. Bắt đầu mới.")
        return state

    with open(path, "r", encoding="utf-8") as f:
        saved = json.load(f)

    n = len(state["questions"])

    # Chỉ khôi phục nếu kích thước khớp
    if len(saved.get("routes", [])) == n:
        state["routes"] = saved["routes"]
        state["execution_codes"] = saved["execution_codes"]
        state["sandbox_results"] = saved["sandbox_results"]
        state["final_answers"] = saved["final_answers"]

        done_count = sum(1 for a in state["final_answers"] if a != "")
        logger.info("[Checkpoint] Đã khôi phục tiến trình: %d/%d câu đã xử lý.", done_count, n)
    else:
        logger.warning("[Checkpoint] Kích thước checkpoint (%d) không khớp dữ liệu (%d). Bỏ qua.",
                       len(saved.get("routes", [])), n)

    return state


# ===========================================================================
# MODULE: pipeline/majority_voting.py
# ===========================================================================
"""Majority Voting — Bầu chọn đáp án cho câu hỏi toán học."""

from collections import Counter
from typing import List


import logging
logger = logging.getLogger("HackAIthon_Agent")


def run_majority_voting(
    state: GraphState, model, tokenizer, indices: List[int], checkpoint_path: str = None
) -> GraphState:
    """
    Majority Voting cho các câu CODEABLE.
    Sandbox khớp → +2 phiếu. LLM 3 vòng → +3 phiếu.
    """
    if not indices:
        return state

    cfg = settings.agents.voting
    vote_batch_size = cfg.batch_size  # = 2 (chống OOM)
    n_votes = cfg.num_runs            # = 3

    logger.info("[Voting] Bắt đầu Majority Voting cho %d câu (batch_size=%d, n_votes=%d)...",
                len(indices), vote_batch_size, n_votes)

    for v_i in range(0, len(indices), vote_batch_size):
        v_end = min(v_i + vote_batch_size, len(indices))
        batch_indices = indices[v_i:v_end]

        # Skip nếu tất cả đã chốt chữ cái đơn
        if all(
            len(state["final_answers"][idx]) == 1
            and state["final_answers"][idx] in "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
            for idx in batch_indices
        ):
            continue

        # Tạo voting prompts
        voting_prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]
            current_status = state["final_answers"][idx]

            if current_status.startswith("SUCCESS_MATCH"):
                _, closest_letter, stdout_val = current_status.split(":", 2)
                python_hint = (
                    f"\n[Kết quả chạy chương trình Python gợi ý]: {stdout_val}"
                    f"\n[Khớp phương án số học tương ứng]: {closest_letter}"
                )
            elif "reasoning" in current_status:
                python_hint = (
                    f"\n[Lưu ý hệ thống]: Sandbox lỗi toán học. "
                    f"Kết quả phân tích suy luận văn bản dự phòng: {current_status}"
                )
            else:
                python_hint = (
                    "\n[Lưu ý hệ thống]: Không thể thực thi mã Python "
                    "hoặc trích xuất suy luận văn bản chuẩn xác."
                )

            prompt = tokenizer.apply_chat_template([
                {
                    "role": "system",
                    "content": (
                        "Bạn là chuyên gia toán học xuất sắc. "
                        "Dựa trên câu hỏi, phương án trắc nghiệm và kết quả gợi ý định lượng, "
                        "hãy chọn đáp án đúng nhất.\n"
                        "Bắt buộc phải trả về định dạng JSON sạch:\n"
                        '{"reasoning": "Suy luận logic ngắn gọn", '
                        '"answer": "Chữ cái lựa chọn đúng duy nhất"}\n'
                        "Tuyệt đối không giải thích thêm ngoài khối JSON."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"/think\nCâu hỏi: {q['question']}\n"
                        f"Lựa chọn:\n{format_choices(q['choices'])}\n"
                        f"{python_hint}"
                    ),
                },
            ], tokenize=False, add_generation_prompt=True)
            voting_prompts.append(prompt)

        # Chạy n_votes vòng bỏ phiếu
        all_votes_matrix = []
        for vote_round in range(n_votes):
            logger.info("   [Voting Round %d/%d] Câu %d đến %d...",
                        vote_round + 1, n_votes, v_i + 1, v_end)
            round_outputs = batch_inference(
                model, tokenizer, voting_prompts,
                max_new_tokens=cfg.max_new_tokens,
                temperature=cfg.temperature,
                micro_batch_size=vote_batch_size,
            )
            all_votes_matrix.append(round_outputs)

        # Đếm phiếu
        for list_pos, idx in enumerate(batch_indices):
            votes = Counter()

            # Phiếu từ LLM (mỗi vòng +1)
            for vote_round in range(n_votes):
                raw_text = all_votes_matrix[vote_round][list_pos]
                parsed_letter = parse_answer_to_letter(raw_text)
                votes[parsed_letter] += 1.0

            # Ưu tiên Sandbox thành công (+2 phiếu)
            if state["final_answers"][idx].startswith("SUCCESS_MATCH"):
                _, closest_letter, _ = state["final_answers"][idx].split(":", 2)
                if closest_letter:
                    votes[closest_letter] += 2.0

            # Chốt đáp án nhiều phiếu nhất
            winner = votes.most_common(1)[0][0]
            state["final_answers"][idx] = winner

        save_checkpoint(state)

    return state


# ===========================================================================
# MODULE: core/llm_engine.py
# ===========================================================================
"""LLM Engine — Load model Unsloth, batch inference, VRAM cleanup."""

import gc
import sys
import time
from types import ModuleType
from typing import List

import torch

# ===========================================================================
# COMPATIBILITY PATCHES (phải chạy TRƯỚC import unsloth)
# ===========================================================================

# Patch 1: torch.float8_e8m0fnu (thiếu trên một số phiên bản PyTorch)
if not hasattr(torch, "float8_e8m0fnu"):
    setattr(torch, "float8_e8m0fnu", torch.float32)

# Patch 2: all_special_tokens_extended (thiếu trên transformers mới)
import transformers
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

if not hasattr(PreTrainedTokenizerBase, "all_special_tokens_extended"):
    PreTrainedTokenizerBase.all_special_tokens_extended = property(
        lambda self: self.all_special_tokens
    )

# Patch 3: pyairports dummy module (thiếu trên Kaggle/Docker)
if "pyairports" not in sys.modules:
    dummy = ModuleType("pyairports")
    dummy_airports = ModuleType("pyairports.airports")
    dummy_airports.AIRPORT_LIST = []
    dummy.airports = dummy_airports
    sys.modules["pyairports"] = dummy
    sys.modules["pyairports.airports"] = dummy_airports

# Patch 4: nest_asyncio (cần cho một số môi trường notebook)
import nest_asyncio
nest_asyncio.apply()

# ===========================================================================
# BÂY GIỜ MỚI import Unsloth
# ===========================================================================
from unsloth import FastLanguageModel


import logging
logger = logging.getLogger("HackAIthon_Agent")


def cleanup_vram():
    """Dọn rác VRAM sau mỗi batch inference."""
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(0.5)


def load_model():
    """Nạp model Unsloth 4-bit và chuẩn bị cho inference."""
    logger.info("Đang nạp mô hình: %s (4-bit, seq_len=%d)...",
                settings.llm.model_name, settings.llm.max_seq_length)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=settings.llm.model_name,
        max_seq_length=settings.llm.max_seq_length,
        load_in_4bit=settings.llm.load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

    logger.info("Nạp mô hình thành công!")
    return model, tokenizer


def batch_inference(
    model,
    tokenizer,
    prompts: List[str],
    max_new_tokens: int = 256,
    temperature: float = 0.2,
    micro_batch_size: int = 10,
) -> List[str]:
    """
    Batch inference phòng thủ: chia prompts thành micro-batches,
    decode chỉ phần generated tokens, dọn VRAM sau mỗi micro-batch.
    """
    results = []

    for i in range(0, len(prompts), micro_batch_size):
        batch = prompts[i: i + micro_batch_size]

        inputs = tokenizer(
            text=batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=settings.llm.max_seq_length,
        ).to("cuda")

        gen_kwargs = {
            "max_new_tokens": max_new_tokens,
            "use_cache": True,
        }
        if temperature == 0.0:
            gen_kwargs["do_sample"] = False
        else:
            gen_kwargs["do_sample"] = True
            gen_kwargs["temperature"] = temperature

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        # Decode CHỈ phần generated tokens (bỏ phần prompt)
        for j, output in enumerate(outputs):
            input_len = inputs.input_ids[j].shape[0]
            generated_tokens = output[input_len:]
            decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            results.append(decoded.strip())

        del inputs, outputs
        cleanup_vram()

    return results


# ===========================================================================
# MODULE: agents/router.py
# ===========================================================================
"""Router Agent — Phân loại câu hỏi thành FAST_QA / READING / CODEABLE."""

import re

from pydantic import BaseModel, Field, ValidationError
from typing import Literal


import logging
logger = logging.getLogger("HackAIthon_Agent")


class RouterSchema(BaseModel):
    route: Literal["FAST_QA", "READING", "CODEABLE"] = Field(
        description="Phân luồng xử lý chính xác cho câu hỏi."
    )


ROUTER_SYSTEM_PROMPT = (
    "Bạn là bộ định tuyến dữ liệu siêu tốc của hệ thống LangGraph. "
    "Nhiệm vụ của bạn là phân loại câu hỏi đầu vào vào một nhóm duy nhất.\n"
    "Bắt buộc phải trả về dữ liệu cấu trúc dưới dạng JSON khớp với schema sau:\n"
    '{"route": "FAST_QA" | "READING" | "CODEABLE"}\n\n'
    "Quy tắc phân loại:\n"
    "1. 'READING': Nếu câu hỏi là đọc hiểu văn bản dài, có các đoạn thông tin "
    "hoặc đoạn trích lịch sử/pháp luật dài phức tạp.\n"
    "2. 'CODEABLE': Nếu câu hỏi yêu cầu giải toán, tính công thức tài chính/kinh tế, "
    "tính số liệu lý/hóa cụ thể.\n"
    "3. 'FAST_QA': Định nghĩa ngắn, tri thức nền hoặc kiểm tra sự thật đơn giản "
    "dưới 2000 ký tự.\n"
    "Chỉ trả ra chuỗi JSON sạch. Tuyệt đối không viết thêm lời dẫn."
)


def router_node(state: GraphState, model, tokenizer) -> GraphState:
    """Node 1: Phân loại batch câu hỏi."""
    cfg = settings.agents.router
    total = len(state["questions"])

    logger.info("\n" + "=" * 75)
    logger.info("[Node 1] Khởi chạy LLM Router Agent")
    logger.info("=" * 75)

    for i in range(0, total, cfg.batch_size):
        end_idx = min(i + cfg.batch_size, total)

        # Skip câu đã route
        if all(state["routes"][k] != "" for k in range(i, end_idx)):
            continue

        logger.info("   [Router] Đang xử lý câu %d đến %d / %d...", i + 1, end_idx, total)

        prompts = []
        for q in state["questions"][i:end_idx]:
            prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"Câu hỏi: {q['question']}\n"
                    f"Lựa chọn:\n{format_choices(q['choices'])}"
                )},
            ], tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        raw_outputs = batch_inference(
            model, tokenizer, prompts,
            max_new_tokens=cfg.max_new_tokens,
            temperature=cfg.temperature,
            micro_batch_size=cfg.batch_size,
        )

        for j, raw_out in enumerate(raw_outputs):
            try:
                json_str = re.search(r'\{.*\}', raw_out, re.DOTALL).group(0)
                validated = RouterSchema.model_validate_json(json_str)
                state["routes"][i + j] = validated.route
            except Exception:
                # Fallback về FAST_QA (KHÔNG PHẢI "A")
                state["routes"][i + j] = "FAST_QA"

        save_checkpoint(state)

    from collections import Counter
    counts = Counter(state["routes"])
    logger.info("[Router] Phân luồng hoàn tất: %s", dict(counts))

    return state


# ===========================================================================
# MODULE: agents/fast_qa.py
# ===========================================================================
"""Fast-QA Agent — Zero-shot trả lời nhanh câu hỏi tri thức nền."""


import logging
logger = logging.getLogger("HackAIthon_Agent")


FAST_QA_SYSTEM_PROMPT = (
    "Bạn là chuyên gia trắc nghiệm tri thức. Hãy phân tích và chọn đáp án đúng duy nhất.\n"
    "Bắt buộc phải trả về định dạng cấu trúc JSON sạch khớp chính xác với mẫu sau:\n"
    '{"reasoning": "Suy luận ngắn gọn (độ dài tối đa 1 câu)", '
    '"answer": "Chữ cái viết hoa duy nhất (Ví dụ: A hoặc B hoặc C...)"}\n\n'
    "QUY TẮC PHÒNG THỦ TRÀN TOKEN:\n"
    "Phần 'reasoning' chỉ được viết đúng 1 câu duy nhất, đi thẳng vào bản chất tri thức nền. "
    "Tuyệt đối không giải thích dông dài, không viết ngoài khối JSON."
)

FAST_QA_FEW_SHOTS = (
    "Ví dụ mẫu:\n"
    "Câu hỏi: Thủ đô của Việt Nam là gì?\n"
    "Lựa chọn:\nA. TP. Hồ Chí Minh\nB. Hà Nội\n"
    "Đầu ra mẫu:\n"
    "```json\n"
    "{\n"
    '  "reasoning": "Hà Nội là thủ đô hành chính chính thức của Việt Nam.",\n'
    '  "answer": "B"\n'
    "}\n"
    "```\n\n"
)


def fast_qa_node(state: GraphState, model, tokenizer) -> GraphState:
    """Node 2: Giải quyết câu hỏi tri thức nền bằng Zero-shot."""
    indices = [i for i, r in enumerate(state["routes"]) if r == "FAST_QA"]
    if not indices:
        return state

    cfg = settings.agents.fast_qa
    total = len(indices)
    logger.info("\n[Node 2] Fast-QA Agent: %d câu tri thức ngắn", total)

    for i in range(0, total, cfg.batch_size):
        end_idx = min(i + cfg.batch_size, total)
        batch_indices = indices[i:end_idx]

        # Skip câu đã trả lời
        if all(state["final_answers"][idx] != "" for idx in batch_indices):
            continue

        logger.info("   [Fast-QA] Câu %d đến %d / %d...", i + 1, end_idx, total)

        prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]
            prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": FAST_QA_SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"/no_think\n{FAST_QA_FEW_SHOTS}"
                    f"BÂY GIỜ HÃY GIẢI CÂU HỎI SAU VÀ TUÂN THỦ JSON THUẬN CÔ ĐỌNG:\n"
                    f"Câu hỏi: {q['question']}\n"
                    f"Lựa chọn:\n{format_choices(q['choices'])}"
                )},
            ], tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        raw_outputs = batch_inference(
            model, tokenizer, prompts,
            max_new_tokens=cfg.max_new_tokens,
            temperature=cfg.temperature,
            micro_batch_size=cfg.batch_size,
        )

        for j, raw_out in enumerate(raw_outputs):
            state["final_answers"][batch_indices[j]] = raw_out

        save_checkpoint(state)

    return state


# ===========================================================================
# MODULE: agents/reading.py
# ===========================================================================
"""Reading Agent — Chain-of-Thought đọc hiểu văn bản dài, rà bẫy phủ định."""


import logging
logger = logging.getLogger("HackAIthon_Agent")


READING_SYSTEM_PROMPT = (
    "Bạn là chuyên gia rà soát bẫy văn bản dài. "
    "Hãy đối chiếu chi tiết các tình huống, rà soát kỹ các từ phủ định "
    "(không phải, ngoại trừ, sai) để trích xuất thông tin.\n"
    "Bắt buộc phải trả về định dạng cấu trúc JSON sạch khớp chính xác với mẫu sau:\n"
    '{"reasoning": "Đối chiếu bối cảnh tài liệu (tối đa 1 đến 2 câu ngắn)", '
    '"answer": "Chữ cái đáp án viết hoa đúng nhất"}\n\n'
    "QUY TẮC CHỐNG CẮT CỤT CHUỖI DO HẾT TOKEN:\n"
    "Phần 'reasoning' phải viết cực kỳ cô đọng, đi thẳng vào việc chỉ rõ "
    "Đoạn/Tài liệu nào chứa từ khóa để chốt đáp án. "
    "Tuyệt đối không chép lại cả đoạn văn bản dài vào JSON."
)

READING_FEW_SHOTS = (
    "Ví dụ mẫu:\n"
    "Câu hỏi: Giai đoạn Mạt Pháp bắt đầu khi nào theo tài liệu?\n"
    "Lựa chọn:\nA. 1000 năm\nB. 1500 năm\n"
    "Đầu ra mẫu:\n"
    "```json\n"
    "{\n"
    '  "reasoning": "Tài liệu tại Đoạn 1 ghi nhận thời điểm Mạt Pháp bắt đầu '
    'là 1500 năm sau khi Thích Ca nhập niết bàn, khớp với phương án B.",\n'
    '  "answer": "B"\n'
    "}\n"
    "```\n\n"
)


def reading_node(state: GraphState, model, tokenizer) -> GraphState:
    """Node 3: Đọc hiểu văn bản dài bằng Chain-of-Thought."""
    indices = [i for i, r in enumerate(state["routes"]) if r == "READING"]
    if not indices:
        return state

    cfg = settings.agents.reading
    total = len(indices)
    logger.info("\n[Node 3] Reading Agent: %d câu đọc hiểu văn bản dài", total)

    for i in range(0, total, cfg.batch_size):
        end_idx = min(i + cfg.batch_size, total)
        batch_indices = indices[i:end_idx]

        # Skip câu đã trả lời
        if all(state["final_answers"][idx] != "" for idx in batch_indices):
            continue

        logger.info("   [Reading] Câu %d đến %d / %d...", i + 1, end_idx, total)

        prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]
            prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": READING_SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"/think\n{READING_FEW_SHOTS}"
                    f"BÂY GIỜ HÃY ĐỐI CHIẾU VĂN BẢN VÀ GIẢI CÂU HỎI SAU:\n"
                    f"Câu hỏi: {q['question']}\n"
                    f"Lựa chọn:\n{format_choices(q['choices'])}"
                )},
            ], tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        raw_outputs = batch_inference(
            model, tokenizer, prompts,
            max_new_tokens=cfg.max_new_tokens,
            temperature=cfg.temperature,
            micro_batch_size=cfg.batch_size,
        )

        for j, raw_out in enumerate(raw_outputs):
            state["final_answers"][batch_indices[j]] = raw_out

        save_checkpoint(state)

    return state


# ===========================================================================
# MODULE: agents/coder.py
# ===========================================================================
"""Coder Agent — Sinh code Python, chạy Sandbox, Self-Correction, Fallback."""

import re


import logging
logger = logging.getLogger("HackAIthon_Agent")


# ===========================================================================
# PROMPTS
# ===========================================================================

CODER_SYSTEM_PROMPT = (
    "Bạn là một chuyên gia lập trình Python tối giản, phụ trách xử lý các bài toán "
    "định lượng và logic đa ngành.\n"
    "Nhiệm vụ: Hãy viết mã nguồn Python hoàn chỉnh để giải bài toán. "
    "Bạn được cung cấp danh sách phương án trắc nghiệm thực tế CHỈ ĐỂ tham khảo "
    "dạng kết quả số học hoặc kí tự đại số.\n\n"
    "QUY TẮC ÉP KHUÔN ĐỊNH DẠNG TUYỆT ĐỐI KHÔNG GÂY LỖI CÚ PHÁP:\n"
    "1. CẤM TUYỆT ĐỐI SỬ DỤNG KÝ TỰ DẤU THĂNG (#). Không viết comment, "
    "không tạo chuỗi hậu tố lặp lại dông dài.\n"
    "2. CẤM TUYỆT ĐỐI IN RA CÁC KÝ TỰ NHÃN ĐÁP ÁN NHƯ 'A', 'B', 'C', 'D'...\n"
    "3. QUY TẮC LỆNH PRINT() CUỐI CÙNG:\n"
    "   - Nếu các phương án lựa chọn chứa số → print(float(gia_tri)). "
    "Tuyệt đối không thêm chuỗi đơn vị.\n"
    "   - Nếu các phương án lựa chọn là biểu thức kí tự/đại số → "
    "print(str(ket_qua)) từ SymPy.\n"
    "4. Toàn bộ mã nguồn phải bọc gọn trong try-except toàn cục. "
    "Nhánh except chỉ viết: print('ERROR').\n"
    "5. Chỉ xuất duy nhất khối mã trong thẻ ```python và ```. Không viết lời dẫn."
)

CODER_FEW_SHOTS = (
    "Ví dụ 1 (Đề bài chứa số hoặc đơn vị kèm theo như %, g, kJ/mol, đô la):\n"
    "Câu hỏi: Tính khối lượng NaOH cần dùng để trung hòa 200ml dung dịch HCl 1M.\n"
    "Lựa chọn đáp án:\nA. 40g.\nB. 80g.\nC. 160g.\n"
    "Mã nguồn mẫu:\n"
    "```python\n"
    "try:\n"
    "    n_hcl = 0.2 * 1.0\n"
    "    m_naoh = n_hcl * 40.0\n"
    "    print(float(m_naoh))\n"
    "except:\n"
    "    print('ERROR')\n"
    "```\n\n"
    "Ví dụ 2 (Đề bài chứa công thức kí tự, phương trình đại số):\n"
    "Câu hỏi: Tìm nghiệm của phương trình vi phân B'(t) = -k*B(t) với B(0)=B0.\n"
    "Lựa chọn đáp án:\nA. B0 * exp(-k * t)\nB. B0 / (1 + k * t)\n"
    "Mã nguồn mẫu:\n"
    "```python\n"
    "try:\n"
    "    import sympy\n"
    "    t, B0, k = sympy.symbols('t B0 k')\n"
    "    ket_qua = B0 * sympy.exp(-k * t)\n"
    "    print(str(ket_qua))\n"
    "except:\n"
    "    print('ERROR')\n"
    "```\n\n"
)

CORRECTION_SYSTEM_PROMPT = (
    "Bạn là một chuyên gia sửa lỗi và tối ưu hóa mã nguồn Python đa ngành.\n"
    "Nhiệm vụ: Viết lại một script Python hoàn chỉnh mới, sửa đổi logic tính toán "
    "hoặc sửa triệt để lỗi gọi tên biến/cú pháp trong hàm print.\n\n"
    "QUY TẮC BẮT BUỘC:\n"
    "1. CẤM TUYỆT ĐỐI SỬ DỤNG KÝ TỰ DẤU THĂNG (#).\n"
    "2. CẤM in ra các chữ cái nhãn đáp án A, B, C, D...\n"
    "3. Kiểm tra kĩ lưỡng tên biến bên trong print() có trùng khớp với biến đã khai báo không.\n"
    "4. Chỉ xuất duy nhất khối mã trong thẻ ```python và ```."
)

FALLBACK_SYSTEM_PROMPT = (
    "Bạn là chuyên gia giải đề trắc nghiệm có tư duy phản biện cao.\n"
    "Thuật toán viết mã giải toán trước đó của hệ thống đã bị tính toán lệch số "
    "hoặc lỗi runtime sau nhiều lần thử.\n"
    "Nhiệm vụ: Dựa vào câu hỏi, các phương án lựa chọn và chi tiết log lỗi, "
    "hãy phân tích lập luận văn bản để tìm ra đáp án đúng cuối cùng.\n"
    "Bắt buộc phải trả về JSON sạch:\n"
    '{"reasoning": "Phân tích logic và suy luận", '
    '"answer": "Chữ cái viết hoa đáp án chuẩn xác nhất"}\n'
    "Tuyệt đối không giải thích thêm ngoài khối JSON."
)


def _extract_code(raw_output: str) -> str:
    """Trích xuất code Python từ output LLM."""
    match = re.search(r'```python\s*(.*?)\s*```', raw_output, re.DOTALL)
    return match.group(1) if match else raw_output


def coder_sandbox_node(state: GraphState, model, tokenizer) -> GraphState:
    """Node 4 & 5: Sinh code → Sandbox → Self-Correction → Fallback → Voting."""
    indices = [i for i, r in enumerate(state["routes"]) if r == "CODEABLE"]
    if not indices:
        return state

    cfg_coder = settings.agents.coder
    cfg_correction = settings.agents.correction
    cfg_fallback = settings.agents.fallback
    sandbox_cfg = settings.sandbox
    total = len(indices)

    logger.info("\n[Node 4&5] Coder & Sandbox Loop: %d câu toán học", total)

    # ===========================================================================
    # CHU TRÌNH 1: SINH MÃ PYTHON
    # ===========================================================================
    for i in range(0, total, cfg_coder.batch_size):
        end_idx = min(i + cfg_coder.batch_size, total)
        batch_indices = indices[i:end_idx]

        # Skip câu đã sinh code
        if all(state["execution_codes"][idx] != "" for idx in batch_indices):
            continue

        logger.info("   [Coder] Sinh code: Câu %d đến %d / %d...", i + 1, end_idx, total)

        prompts = []
        for idx in batch_indices:
            q = state["questions"][idx]
            prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": CODER_SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"/no_think\n{CODER_FEW_SHOTS}"
                    f"BÂY GIỜ HÃY GIẢI CÂU HỎI SAU, CHÚ Ý PRINT KẾT QUẢ SỐ THUẦN TUÝ "
                    f"HOẶC CHUỖI BIỂU THỨC SẠCH:\n"
                    f"Câu hỏi: {q['question']}\n"
                    f"Lựa chọn đáp án thực tế:\n{format_choices(q['choices'])}"
                )},
            ], tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        raw_codes = batch_inference(
            model, tokenizer, prompts,
            max_new_tokens=cfg_coder.max_new_tokens,
            temperature=cfg_coder.temperature,
            micro_batch_size=cfg_coder.batch_size,
        )

        for j, raw_code in enumerate(raw_codes):
            state["execution_codes"][batch_indices[j]] = _extract_code(raw_code)

        save_checkpoint(state)

    # ===========================================================================
    # CHU TRÌNH 2: THỰC THI SANDBOX & SELF-CORRECTION
    # ===========================================================================
    max_retries = sandbox_cfg.max_retries  # = 2

    for attempt in range(max_retries):
        # Tìm câu chưa thành công
        active_indices = []
        for idx in indices:
            ans = state["final_answers"][idx]
            if (ans.startswith("SUCCESS_MATCH")
                    or (len(ans) == 1 and ans in "ABCDEFGHIJKLMNOPQRSTUVWXYZ")
                    or "reasoning" in ans):
                continue
            active_indices.append(idx)

        if not active_indices:
            break

        logger.info("   [Sandbox] Thực thi lượt %d/%d: %d câu...",
                    attempt + 1, max_retries, len(active_indices))

        correction_prompts_map = {}
        next_active = []

        for prog_idx, idx in enumerate(active_indices):
            code = state["execution_codes"][idx]
            logger.info("      [Sandbox Run] (%d/%d) Index %d...",
                        prog_idx + 1, len(active_indices), idx)

            # Chạy code
            run_res = execute_code(code, timeout=sandbox_cfg.timeout_sec)
            state["sandbox_results"][idx] = run_res

            # So khớp kết quả
            closest_letter, min_diff, target_val = None, float("inf"), None
            if run_res["success"]:
                closest_letter, min_diff, target_val = find_closest_choice_info(
                    run_res["stdout"], state["questions"][idx]["choices"]
                )

            # Thành công: sai số < 1%
            if run_res["success"] and closest_letter and min_diff < 0.01:
                state["final_answers"][idx] = (
                    f"SUCCESS_MATCH:{closest_letter}:{run_res['stdout']}"
                )
            else:
                # Tạo thông báo lỗi chi tiết
                if not run_res["success"]:
                    err_msg = f"Mã nguồn bị lỗi Runtime.\nLog: {run_res['stderr']}"
                elif target_val is None and run_res["stdout"] == "ERROR":
                    err_msg = "Mã nguồn crash nhảy vào except do lỗi cú pháp hoặc biến."
                elif target_val is None:
                    err_msg = (f"Mã chạy thành công nhưng stdout không khớp dải đáp án "
                               f"(stdout: '{run_res['stdout']}').")
                else:
                    err_msg = (f"Mã chạy ra '{target_val}' nhưng lệch quá xa "
                               f"(sai số nhỏ nhất: {min_diff * 100:.2f}% tại {closest_letter}).")

                # Còn lượt: tạo prompt sửa code
                if attempt < max_retries - 1:
                    cor_prompt = tokenizer.apply_chat_template([
                        {"role": "system", "content": CORRECTION_SYSTEM_PROMPT},
                        {"role": "user", "content": (
                            f"/no_think\n"
                            f"Hãy sửa đổi và viết lại đoạn mã nguồn mới hoàn chỉnh.\n\n"
                            f"Câu hỏi: {state['questions'][idx]['question']}\n"
                            f"Lựa chọn:\n{format_choices(state['questions'][idx]['choices'])}\n\n"
                            f"Thông báo lỗi:\n{err_msg}\n\n"
                            f"Code lỗi:\n{code}"
                        )},
                    ], tokenize=False, add_generation_prompt=True)
                    correction_prompts_map[idx] = cor_prompt
                    next_active.append(idx)
                else:
                    # Hết lượt → Fallback Agent
                    logger.info("      [Fallback] Index %d đã thử sai %d lần. Suy luận văn bản...",
                                idx, max_retries)
                    fb_prompt = tokenizer.apply_chat_template([
                        {"role": "system", "content": FALLBACK_SYSTEM_PROMPT},
                        {"role": "user", "content": (
                            f"Câu hỏi: {state['questions'][idx]['question']}\n"
                            f"Lựa chọn:\n{format_choices(state['questions'][idx]['choices'])}\n\n"
                            f"Mã nguồn lỗi:\n{code}\n\n"
                            f"Log lỗi:\n{err_msg}"
                        )},
                    ], tokenize=False, add_generation_prompt=True)

                    fb_out = batch_inference(
                        model, tokenizer, [fb_prompt],
                        max_new_tokens=cfg_fallback.max_new_tokens,
                        temperature=cfg_fallback.temperature,
                        micro_batch_size=1,
                    )
                    state["final_answers"][idx] = fb_out[0]

        save_checkpoint(state)

        # Chạy LLM Correction cho các câu cần sửa
        if next_active and attempt < max_retries - 1:
            logger.info("   [Correction] Vá lỗi %d câu...", len(next_active))
            cor_batch = cfg_correction.batch_size
            for c_i in range(0, len(next_active), cor_batch):
                c_end = min(c_i + cor_batch, len(next_active))
                sub_batch = next_active[c_i:c_end]

                logger.info("      [Correction] Câu %d đến %d / %d...",
                            c_i + 1, c_end, len(next_active))

                prompts_batch = [correction_prompts_map[idx] for idx in sub_batch]
                raw_cor_codes = batch_inference(
                    model, tokenizer, prompts_batch,
                    max_new_tokens=cfg_correction.max_new_tokens,
                    temperature=cfg_correction.temperature,
                    micro_batch_size=cor_batch,
                )

                for j, raw_code in enumerate(raw_cor_codes):
                    state["execution_codes"][sub_batch[j]] = _extract_code(raw_code)

                save_checkpoint(state)

    return state


# ===========================================================================
# MODULE: main.py
# ===========================================================================
"""Entry Point — Khởi tạo mô hình, nạp dữ liệu và điều phối Multi-Agent Graph."""

import os
import time





def main():
    start_time = time.time()
    logger.info("===========================================================================")
    logger.info("BẮT ĐẦU CHẠY PIPELINE MULTI-AGENT HACKAITHON 2026")
    logger.info("===========================================================================")

    # 1. Khởi tạo môi trường
    os.makedirs(settings.paths.output_dir, exist_ok=True)

    # 2. Nạp mô hình LLM Unsloth
    model, tokenizer = load_model()

    # 3. Nạp dữ liệu
    questions = load_test_data()
    state = init_state(questions)

    # 4. Nạp checkpoint cũ (nếu có)
    state = load_checkpoint(state)

    # 5. Node 1: Router Agent (Phân luồng)
    state = router_node(state, model, tokenizer)

    # 6. Node 2: Fast-QA Agent (Tri thức nền)
    state = fast_qa_node(state, model, tokenizer)

    # 7. Node 3: Reading Comprehension Agent (Đọc hiểu)
    state = reading_node(state, model, tokenizer)

    # 8. Node 4 & 5: Coder & Sandbox Loop (Toán học & Code)
    state = coder_sandbox_node(state, model, tokenizer)

    # 9. Node 5 (Voting): Majority Voting cho các câu CODEABLE
    codeable_indices = [i for i, r in enumerate(state["routes"]) if r == "CODEABLE"]
    if codeable_indices:
        state = run_majority_voting(state, model, tokenizer, codeable_indices)

    # 10. Node 6: Answer Voter & Dynamic Mapper (Chuẩn hoá đáp án)
    state = map_final_answers(state)

    # 11. Ghi file kết quả pred.csv và lưu checkpoint cuối cùng
    save_predictions(state)
    save_checkpoint(state)

    total_time = time.time() - start_time
    logger.info("===========================================================================")
    logger.info("HOÀN THÀNH PIPELINE TRONG %.2f PHÚT (%.2f GIÂY)", total_time / 60, total_time)
    logger.info("===========================================================================")


if __name__ == "__main__":
    main()




### 4. Khởi chạy Pipeline trên Kaggle


In [ ]:
import os

# Cấu hình đường dẫn đầu ra làm việc
os.makedirs('/kaggle/working/output', exist_ok=True)

# Tự động tìm file dữ liệu (.csv hoặc .json) trong các datasets đã add vào Kaggle
input_dir = '/kaggle/input'
found_file = None

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if file.endswith('.json') or file.endswith('.csv'):
            found_file = os.path.join(root, file)
            break
    if found_file:
        break

if found_file:
    print(f'[Kaggle] Tìm thấy dữ liệu tại: {found_file}')
    # Ghi đè cấu hình Settings để trỏ tới thư mục chứa dữ liệu
    settings.paths.data_dir = os.path.dirname(found_file)
    settings.paths.output_dir = '/kaggle/working/output'
    # Gọi hàm main() chạy toàn bộ pipeline
    main()
else:
    print('⚠️ Cảnh báo: Không tìm thấy file dữ liệu nào trong /kaggle/input.')
    print('Hãy chắc chắn rằng bạn đã bấm "Add Input" ở góc phải và thêm dataset thi đấu.')
